# Python 문법 파일별 6 · 24-1 같은 위치끼리 묶기 · `zip`

**한 줄 요약** · 리스트 3개를 `zip`으로 묶어 딕셔너리를 세 방법으로 만들고, `strict` · 풀기 · 라벨 짝짓기까지 정리

- **원본** · `1.Python문법/01_실습문제/24-1.Parallel Iteration_Q.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/01_Python문법` 전체 정리 3-4 (리스트로 딕셔너리 만들기)

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. `zip(a, b, c)`가 만드는 묶음을 실행 전에 말한다
2. 같은 결과를 컴프리헨션 · 반복문 · `dict(zip())` 세 방법으로 만든다
3. 길이가 다른 리스트를 `zip`할 때의 결과와 `strict=True`를 설명한다
4. `enumerate`와 `zip`을 함께 쓰고, `zip(*묶음)`으로 다시 푼다

## 어디에 쓰나

*실제 상황 → 이 권에서 배우는 것*

- **이미지 · 라벨 리스트를 짝지을 때** · `zip(paths, labels, strict=True)` (2-1 · 3장)
- **데이터로더에서 배치를 꺼낼 때** · `for images, labels in loader:` — 같은 짝 풀기
- **클래스 이름 → 번호 사전을 만들 때** · `dict(zip(names, range(n)))` (1-2)
- **(입력, 정답) 묶음을 다시 나눌 때** · `zip(*pairs)` (2-2)

## 학습 지도

<img src="images/pf6_map.png" alt="24-1 학습 지도" width="763">

**예상 학습 시간** · 개념 40분 + 실습 20분

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">장</th><th style="text-align: left;">제목</th><th style="text-align: left;">이 장의 질문</th></tr>
</thead>
<tbody>
<tr><td>0</td><td>준비</td><td>실행 환경 · 원본 문제</td></tr>
<tr><td>1</td><td>zip으로 묶어 딕셔너리 만들기</td><td>여러 리스트의 같은 위치 값을 어떻게 묶어 딕셔너리로 만드는가?</td></tr>
<tr><td>2</td><td>zip을 안전하게 쓰기</td><td>길이가 다를 때는 어떻게 되고, 번호 붙이기와 다시 풀기는 어떻게 하는가?</td></tr>
<tr><td>3</td><td>종합 실습</td><td>이미지 경로 · 라벨 리스트를 안전하게 짝지어 데이터 목록을 만들려면?</td></tr>
<tr><td>—</td><td>마무리 · 부록</td><td>오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td></tr>
</tbody></table>

---
## 0. 준비

이 노트북은 라이브러리 없이 파이썬 기본 문법만 씀. **셀은 위에서부터 차례로 실행**(1-1의 `a` · `b` · `c`를 계속 씀).

*항목 → 설정*

- **라이브러리** · 없음
- **seed · 장치** · 필요 없음
- **데이터** · 원본 문제의 리스트 3개 · 3장은 가짜 이미지 경로 · 라벨

**실행 흐름**

1. `sys` 모듈 불러오기
2. 파이썬 버전 출력
3. 3.10 이상인지 확인

In [1]:
# 실행 환경 확인: 이 노트북은 파이썬 기본 문법만 씀
# 왜 · 버전 차이로 출력이 달라지지 않는지 먼저 확인하기 위해
import sys  # 파이썬 버전 확인용

print("Python:", sys.version.split()[0])  # 3.10 이상이면 됨

Python: 3.14.6


### 원본 문제

- 리스트 3개를 `{a의 값: b × c}` 형태의 **딕셔너리**로 바꾸는 문제 · 여러 방법 활용

```python
a = ["one", "two", "three", "four"]
b = [30, 20, 15, 75]
c = [5.2, 7.4, 3.6, 4.2]
```

- 출력 결과는 `{'one': 156.0, 'two': 148.0, 'three': 54.0, 'four': 315.0}`

**원본 힌트**

- 여러 반복 가능한 묶음을 함께 묶어 계산하는 과정이 중요
- `zip` = 여러 그룹의 반복 가능한 자료를 같은 위치끼리 묶는 기능
- 사용법 `zip(*iterables, strict=False)` · `strict`는 Python 3.10에서 추가

---
## 1. zip으로 묶어 딕셔너리 만들기

> **이 장의 흐름**
>
> - `zip`이 만드는 묶음을 먼저 눈으로 보고
> - 원본 정답을 세 가지 방법으로 만들어 비교

<img src="images/pf_zip3.png" alt="리스트 3개를 같은 위치끼리 묶어 딕셔너리로" width="763">

**파이프라인 위치** · ② 불러오기 · ③ 전처리 — 이미지 · 라벨 짝

### 1-1. `A` `zip`이 만드는 묶음 보기 · `zip()` · `list()` <sub>원본 문제 · 힌트</sub>

> 🎯 **`zip(a, b, c)`는 무엇을 하나씩 내주는가?**

💡 **쉽게 말하면**

- `zip`은 **줄 선 사람들을 같은 위치끼리 짝짓기**.
- 세 줄을 짝지으면 `(이름, 수량, 단가)` 묶음이 차례로 나옴.

**실행 흐름**

1. 리스트 3개
2. zip으로 묶기
3. list()로 꺼내 보기

**먼저 예측** · `zip(a, b, c)`의 첫 묶음은? — 답은 코드 실행 결과 아래에

In [2]:
# 왜 · 같은 위치끼리 짝이 맞는 세 리스트를 원본 문제 그대로 준비하기 위해
a = ["one", "two", "three", "four"]  # 키가 될 이름
b = [30, 20, 15, 75]                 # 수량
c = [5.2, 7.4, 3.6, 4.2]             # 단가


# 왜 · zip이 같은 위치끼리 묶는다는 것을 list로 펼쳐 눈으로 확인하기 위해
pairs = list(zip(a, b, c))              # 같은 위치끼리 (이름, 수량, 단가)
print(len(pairs), pairs[0], pairs[-1])  # 묶음 수 · 처음 · 마지막
print(type(zip(a, b)))                  # list로 감싸지 않으면 내용이 안 보임

4 ('one', 30, 5.2) ('four', 75, 4.2)
<class 'zip'>


**결과** · 예측 정답은 `('one', 30, 5.2)`

- `4 ('one', 30, 5.2) ('four', 75, 4.2)` — 같은 위치끼리 4묶음
- `<class 'zip'>` — zip은 묶음을 하나씩 내주는 반복 객체 → `list()`로 감싸야 보임

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>zip(a, b, c)</code></td><td><strong>같은 위치끼리</strong> 튜플</td><td>리스트 여러 개를 짝지을 때</td></tr>
<tr><td><code>list(zip(...))</code></td><td>묶음을 한 번에 리스트로</td><td>눈으로 확인할 때</td></tr>
<tr><td><code>for k, x, y in zip(...)</code></td><td>묶음을 바로 풀어 받기</td><td>반복하며 계산할 때 (기본 선택)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - zip 객체는 **한 번 돌면 비워짐** → 두 번 쓰려면 `list()`로 저장

> ✅ **한 줄 정리** · `zip`은 여러 리스트의 같은 위치 값을 튜플로 묶어 하나씩 내준다.

**딥러닝 코드에서는**

- `for images, labels in loader:` — 이미지 · 라벨 짝을 풀어 받기

### 1-2. `A` 정답 · 세 가지 방법 · `{k: v for}` · `dict(zip())` <sub>원본 문제 · 여러 방법</sub>

> 🎯 **같은 딕셔너리를 세 가지 방법으로 만들려면?**

💡 **쉽게 말하면**

- 묶음마다 `수량 × 단가`를 계산해 `{이름: 금액}`으로 담음.
- 한 줄(컴프리헨션) · 여러 줄(반복문) · 짝짓기(`dict(zip)`).

**실행 흐름**

1. 딕셔너리 컴프리헨션
2. 반복문으로 하나씩
3. 금액 리스트 → dict(zip())
4. 세 결과 비교

**먼저 예측** · 결과의 값은 정수일까 실수일까? — 답은 코드 실행 결과 아래에

In [3]:
# 방법 1: 딕셔너리 컴프리헨션
# 왜 · 세 리스트를 한 줄로 묶어 이름별 금액을 바로 만들기 위해
r1 = {k: x * y for k, x, y in zip(a, b, c)}  # 묶음마다 이름: 수량 × 단가


# 방법 2: 반복문으로 하나씩 넣기
# 왜 · 컴프리헨션이 하는 일을 단계별로 풀어 비교하기 위해
r2 = {}                       # 빈 딕셔너리
for k, x, y in zip(a, b, c):  # 묶음 풀기
    r2[k] = x * y             # 키 = 값으로 추가


# 방법 3: 금액 리스트를 먼저 만들고 dict(zip(키, 값))
# 왜 · 키 리스트와 값 리스트를 따로 만든 뒤 합치는 방식도 비교하기 위해
r3 = dict(zip(a, [x * y for x, y in zip(b, c)]))  # 금액 리스트 → 이름과 짝


# 왜 · 세 방법의 결과가 같다는 것을 코드로 확인하기 위해
assert r1 == r2 == r3  # 세 방법 결과가 같은지
print(r1)              # 원본 출력 결과

{'one': 156.0, 'two': 148.0, 'three': 54.0, 'four': 315.0}


**결과** · 예측 정답은 실수 — 단가가 실수라 `156.0`처럼

- `{'one': 156.0, 'two': 148.0, 'three': 54.0, 'four': 315.0}` — 원본 출력 결과와 같음 (30×5.2 · 20×7.4 · 15×3.6 · 75×4.2)
- assert 통과 · 세 방법 모두 같은 딕셔너리

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>{k: v for ...}</code></td><td>변환 · 조건을 <strong>함께</strong> 적용</td><td>값을 계산해서 넣을 때 (기본 선택)</td></tr>
<tr><td>반복문 + <code>d[k] = v</code></td><td>중간에 출력 · 조건을 넣기 쉬움</td><td>계산 과정이 길 때</td></tr>
<tr><td><code>dict(zip(키, 값))</code></td><td>이미 있는 두 리스트를 <strong>그대로</strong> 짝지음</td><td>키 · 값 리스트가 준비돼 있을 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 키 리스트에 같은 값이 두 번 있으면 뒤의 값이 앞을 덮어써 딕셔너리 길이가 줄어듦

> ✅ **한 줄 정리** · `zip`으로 묶은 뒤 컴프리헨션 · 반복문 · `dict(zip())` 중 하나로 `{키: 값}` 딕셔너리를 만든다.

**딥러닝 코드에서는**

- `dict(zip(class_names, range(len(class_names))))` — 클래스 이름 → 번호

---
## 2. zip을 안전하게 쓰기

> **이 장의 흐름**
>
> - 원본 힌트: `zip(*iterables, strict=False)`
> - 길이 차이 · 번호와 함께 · `zip(*묶음)`으로 풀기

<img src="images/pf6_strict.png" alt="길이가 다르면 zip은 잘리고 strict는 에러" width="763">

**파이프라인 위치** · ③ 전처리 — 데이터 짝 확인

### 2-1. `A` 길이가 다를 때 · `strict=True` · `ValueError` <sub>원본 힌트 · strict</sub>

> 🎯 **길이가 다른 리스트를 `zip`하면 어떻게 되는가?**

💡 **쉽게 말하면**

- 짝이 없는 사람은 **조용히 빠짐** (에러 없음).
- `strict=True`면 짝이 안 맞을 때 **에러로 알려 줌**.

**실행 흐름**

1. 길이 3과 2를 zip
2. strict=True로 다시

**먼저 예측** · `zip([1, 2, 3], ["x", "y"])`의 결과는? — 답은 코드 실행 결과 아래에

In [4]:
# 왜 · 길이가 다르면 zip이 조용히 잘라 버리는 위험을 strict=True로 잡기 위해
print(list(zip([1, 2, 3], ["x", "y"])))            # 짧은 쪽에 맞춰 조용히 잘림
try:
    list(zip([1, 2, 3], ["x", "y"], strict=True))  # 길이가 다르면 에러로 알림
except ValueError as e:                            # Python 3.10 이상
    print("ValueError:", e)                        # 에러 문장 출력

[(1, 'x'), (2, 'y')]
ValueError: zip() argument 2 is shorter than argument 1


**결과** · 예측 정답은 `[(1, 'x'), (2, 'y')]` — 3은 사라짐

- `[(1, 'x'), (2, 'y')]` — 3은 짝이 없어 사라짐 (에러 없음)
- `ValueError: zip() argument 2 is shorter than argument 1` — `strict=True`면 길이 차이를 에러로 알려 줌

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>zip(a, b)</code></td><td>짧은 쪽에 맞춰 <strong>잘림</strong></td><td>길이가 달라도 되는 경우</td></tr>
<tr><td><code>zip(a, b, strict=True)</code></td><td>길이가 다르면 <strong>에러</strong></td><td>길이가 꼭 같아야 할 때 (이미지 ↔ 라벨)</td></tr>
<tr><td><code>len(a) == len(b)</code> 확인</td><td>직접 비교</td><td>Python 3.9 이하</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 중간 이미지 1장이 빠지면 그 뒤 짝이 모두 한 칸씩 밀리고 마지막 라벨만 버려짐 → 에러가 없어 원인 찾기 어려움

> ✅ **한 줄 정리** · `zip`은 짧은 쪽에 맞춰 조용히 잘리므로, 길이가 같아야 하면 `strict=True`를 쓴다.

**딥러닝 코드에서는**

- `zip(image_paths, labels, strict=True)` — 데이터 짝 검사

### 2-2. `B` 번호와 함께 · 다시 풀기 · `enumerate(zip())` · `zip(*pairs)` <sub>원본 힌트 · *iterables</sub>

> 🎯 **묶음에 번호를 붙이고, 묶음을 다시 리스트 여러 개로 나누려면?**

💡 **쉽게 말하면**

- `enumerate(zip(...))`는 **번호 + 묶음**.
- `zip(*묶음)`은 묶음을 **다시 줄별로** 나눔 (짝짓기의 반대).

**실행 흐름**

1. enumerate(zip())로 번호
2. zip(*pairs)로 풀기
3. 풀린 결과 확인

**먼저 예측** · `zip(*pairs)`의 첫 결과는? — 답은 코드 실행 결과 아래에

In [5]:
# 왜 · 묶음 번호가 필요할 때 enumerate와 zip을 함께 쓰는 법을 보이기 위해
for i, (k, x) in enumerate(zip(a, b), start=1):  # 번호 + (이름, 수량)
    print(i, k, x)                               # 몇 번째 묶음인지


# 왜 · zip(*묶음)으로 묶음을 다시 줄별로 푸는 방법을 보이기 위해
pairs = list(zip(a, b))      # [('one', 30), ...]
names, counts = zip(*pairs)  # *로 묶음을 풀어 다시 zip → 줄별로
print(names, counts)         # 튜플 두 개

1 one 30
2 two 20
3 three 15
4 four 75
('one', 'two', 'three', 'four') (30, 20, 15, 75)


**결과** · 예측 정답은 `('one', 'two', 'three', 'four')` — 이름만 모임

- `enumerate`로 1부터 번호 + 묶음
  - `1 one 30`
  - `4 four 75`
- `('one', 'two', 'three', 'four') (30, 20, 15, 75)` — `zip(*pairs)`로 다시 이름 줄 · 수량 줄

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>enumerate(zip(a, b))</code></td><td>번호 + 묶음</td><td>몇 번째인지 필요할 때</td></tr>
<tr><td><code>zip(*pairs)</code></td><td>묶음 → <strong>줄별</strong> 튜플</td><td>(입력, 정답) 묶음을 나눌 때</td></tr>
<tr><td><code>for i, (k, x) in ...</code></td><td>괄호로 안쪽 묶음까지 풀기</td><td>번호와 묶음을 한 번에</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `for i, k, x in enumerate(zip(a, b))`처럼 괄호를 빼면 ValueError (2개짜리를 3개로 풀기)

> ✅ **한 줄 정리** · `enumerate(zip(...))`로 번호를 붙이고, `zip(*묶음)`으로 묶음을 다시 줄별로 나눈다.

**딥러닝 코드에서는**

- `images, labels = zip(*batch)` — 데이터로더의 collate 함수에서 자주 씀

---
## 3. 종합 실습 · 이미지 경로와 라벨 짝짓기

> **이 장의 흐름**
>
> - 1 · 2장 문법을 한 번에 사용
> - 데이터셋 준비 단계를 가짜 경로로 흉내 냄

<img src="images/pf6_practice.png" alt="경로 · 라벨을 짝짓고 클래스 번호로 바꾸기" width="763">

**파이프라인 위치** · ② 불러오기 → ③ 전처리

> 🎯 **경로 리스트와 라벨 리스트를 짝짓고, 라벨 이름을 번호로 바꾼 데이터 목록을 만들려면?**

💡 **쉽게 말하면**

- `strict=True`로 **짝 검사**, `dict(zip)`으로 **이름 → 번호**.
- 마지막에 `enumerate`로 몇 번째 샘플인지 출력.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">단계</th><th style="text-align: left;">하는 일</th><th style="text-align: left;">쓰는 문법</th></tr>
</thead>
<tbody>
<tr><td>1</td><td>경로 · 라벨 리스트</td><td>가짜 데이터</td></tr>
<tr><td>2</td><td>클래스 이름 → 번호</td><td>1-2 <code>dict(zip())</code></td></tr>
<tr><td>3</td><td>짝 검사하며 묶기</td><td>2-1 <code>strict=True</code></td></tr>
<tr><td>4</td><td>번호 붙여 출력 · 다시 풀기</td><td>2-2 <code>enumerate</code> · <code>zip(*)</code></td></tr>
</tbody></table>

**먼저 예측** · `class_to_idx["dog"]`는? — 답은 코드 실행 결과 아래에

In [6]:
# 1단계: 가짜 이미지 경로와 라벨 (실제로는 폴더에서 읽음)
# 왜 · 이미지 데이터셋을 만드는 실제 상황을 작은 목록으로 재현하기 위해
paths = ["img/cat1.jpg", "img/dog1.jpg", "img/cat2.jpg", "img/bird1.jpg"]  # 이미지 경로
labels = ["cat", "dog", "cat", "bird"]                                     # 같은 위치의 정답


# 2단계: 클래스 이름 → 번호 (정렬해서 항상 같은 번호)
# 왜 · 실행할 때마다 클래스 번호가 바뀌지 않게 하기 위해
classes = sorted(set(labels))                           # 중복 제거 후 정렬
class_to_idx = dict(zip(classes, range(len(classes))))  # 이름과 번호 짝
print(class_to_idx)                                     # {'bird': 0, 'cat': 1, 'dog': 2}


# 3단계: 길이를 검사하며 (경로, 번호) 묶기
# 왜 · 경로와 라벨 수가 다르면 바로 에러로 알기 위해
data = [(p, class_to_idx[y]) for p, y in zip(paths, labels, strict=True)]  # 짝이 안 맞으면 에러


# 4단계: 번호 붙여 출력하고 다시 풀기
# 왜 · 만든 (경로, 번호) 묶음을 확인하고 다시 두 목록으로 되돌리기 위해
for i, (p, y) in enumerate(data):  # 0부터 샘플 번호
    print(i, p, y)                 # 경로와 라벨 번호
xs, ys = zip(*data)                # 경로 줄 · 라벨 줄로 나누기
print("라벨 줄:", ys)              # 모델에 넣을 정답

{'bird': 0, 'cat': 1, 'dog': 2}
0 img/cat1.jpg 1
1 img/dog1.jpg 2
2 img/cat2.jpg 1
3 img/bird1.jpg 0
라벨 줄: (1, 2, 1, 0)


**결과** · 예측 정답은 `2` — bird · cat · dog 순서

- `{'bird': 0, 'cat': 1, 'dog': 2}` — 정렬된 이름 순서대로 0부터 번호
- strict 검사 통과 · (경로, 번호) 짝
  - `0 img/cat1.jpg 1`
  - `3 img/bird1.jpg 0`
- `라벨 줄: (1, 2, 1, 0)` — `zip(*data)`로 라벨만 모음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>sorted(set(labels))</code></td><td>중복 제거 + 항상 같은 순서</td><td>클래스 목록 만들기</td></tr>
<tr><td><code>dict(zip(classes, range(n)))</code></td><td>이름 → 번호</td><td>라벨 인코딩</td></tr>
<tr><td><code>zip(..., strict=True)</code></td><td>짝 수 검사</td><td>경로 ↔ 라벨</td></tr>
<tr><td><code>zip(*data)</code></td><td>묶음 → 줄별</td><td>입력 · 정답 분리</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `set(labels)`만 쓰면 실행마다 순서가 달라질 수 있음 → 번호가 바뀌지 않게 `sorted`

> ✅ **한 줄 정리** · 경로 · 라벨은 `zip(strict=True)`로 짝짓고, 클래스 이름은 `dict(zip())`으로 번호로 바꾼다.

**딥러닝 코드에서는**

- torchvision `ImageFolder`의 `class_to_idx`가 같은 방식(정렬된 폴더 이름 → 번호)

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">학습 목표</th><th style="text-align: left;">할 수 있게 된 것</th><th style="text-align: left;">확인 위치</th></tr>
</thead>
<tbody>
<tr><td>1. zip 묶음</td><td>같은 위치끼리 튜플 4묶음 · list()로 확인</td><td>1-1</td></tr>
<tr><td>2. 정답 세 방법</td><td>컴프리헨션 · 반복문 · dict(zip()) 결과 같음</td><td>1-2</td></tr>
<tr><td>3. 길이 차이</td><td>zip은 잘림 · strict=True는 ValueError</td><td>2-1</td></tr>
<tr><td>4. 번호 · 풀기</td><td><code>enumerate(zip())</code> · <code>zip(*pairs)</code></td><td>2-2</td></tr>
<tr><td>5. 이미지 · 라벨</td><td>class_to_idx · (경로, 번호) 목록</td><td>3</td></tr>
</tbody></table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td><code>A</code></td><td><code>zip()</code></td><td>같은 위치끼리 묶기</td><td><code>for images, labels in loader</code></td></tr>
<tr><td><code>A</code></td><td><code>strict=True</code></td><td>길이가 다르면 에러</td><td>경로 ↔ 라벨 검사</td></tr>
<tr><td><code>A</code></td><td><code>dict(zip())</code></td><td>키 리스트 · 값 리스트 짝</td><td><code>class_to_idx</code></td></tr>
<tr><td><code>B</code></td><td><code>zip(*pairs)</code></td><td>묶음 → 줄별</td><td><code>images, labels = zip(*batch)</code></td></tr>
</tbody></table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

*섹션 → 🎯 질문*

- **1-1** · `list(zip(a, b, c))[0]`은?
- **1-2** · 원본 정답을 한 줄로 쓰면?
- **2-1** · 길이 3과 2를 zip하면?
- **2-2** · `zip(*[(1, 'a'), (2, 'b')])`의 결과는?
- **3** · 클래스 번호를 항상 같게 하려면?

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `names = ["cat", "dog"]`, `ages = [3, 5]`로 `{이름: 나이}` 딕셔너리 만들기
2. `a`, `b`로 `{이름: 수량 × 2}` 만들기

In [7]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. `b`와 `c`로 금액 합계 구하기 (`sum` + `zip`)
2. `a`의 이름마다 1부터 번호를 붙여 `{번호: 이름}` 만들기
3. `[("x", 1), ("y", 2)]`를 이름 줄과 숫자 줄로 나누기

In [8]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

*섹션 → 답*

- **1-1** · `('one', 30, 5.2)`
- **1-2** · `{k: x * y for k, x, y in zip(a, b, c)}`
- **2-1** · 2묶음으로 잘림 (strict=True면 ValueError)
- **2-2** · `(1, 2)`와 `('a', 'b')`
- **3** · `sorted(set(labels))`로 정렬 후 번호

#### 코드 문제 답

```python
names, ages = ["cat", "dog"], [3, 5]
print(dict(zip(names, ages)))  # {'cat': 3, 'dog': 5}
a, b = ["one", "two", "three", "four"], [30, 20, 15, 75]
print({k: x * 2 for k, x in zip(a, b)})  # {'one': 60, 'two': 40, 'three': 30, 'four': 150}
```

#### 변형 문제 답

*문제 → 답*

- **1** · `sum(x * y for x, y in zip(b, c))` → `673.0`
- **2** · `dict(enumerate(a, start=1))` → `{1: 'one', 2: 'two', 3: 'three', 4: 'four'}`
- **3** · `zip(*[("x", 1), ("y", 2)])` → `('x', 'y')` · `(1, 2)`

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">틀린 내용</th><th style="text-align: left;">틀린 이유</th><th style="text-align: left;">올바른 원칙</th><th style="text-align: left;">재발 방지 질문</th></tr>
</thead>
<tbody>
<tr><td>(예) 라벨 하나가 빠졌는데 에러 없이 학습됨</td><td>zip이 길이가 다르면 에러를 낼 거라 착각</td><td>zip은 짧은 쪽에 맞춰 조용히 잘림</td><td><code>strict=True</code>를 붙였나?</td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
</tbody></table>

---
## 면접 30초 답변

**Q1. `zip`은 무엇을 하나요?**
> - 여러 반복 가능한 자료의 같은 위치 값을 튜플로 묶어 하나씩 내줍니다.
> - 리스트가 아니라 반복 객체이므로 내용을 보려면 `list()`로 감싸야 하고 한 번 돌면 비워집니다.

**Q2. 길이가 다른 리스트를 `zip`하면 어떻게 되나요?**
> - 가장 짧은 쪽에 맞춰 나머지는 에러 없이 버려집니다.
> - 이미지와 라벨처럼 길이가 반드시 같아야 하면 `strict=True`를 붙입니다.
> - 그러면 길이가 다를 때 ValueError가 납니다(Python 3.10 이상).

**Q3. `zip(*pairs)`는 언제 쓰나요?**
> - (입력, 정답) 같은 묶음 리스트를 입력 줄과 정답 줄로 다시 나눌 때 씁니다.
> - 데이터로더의 collate 함수에서 `images, labels = zip(*batch)`처럼 자주 보입니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>zip()</code> · <code>list()</code></td><td>Python 내장 함수</td><td>같은 위치끼리 묶기 · 확인</td><td>1-1 <code>list(zip(a, b, c))</code></td></tr>
<tr><td>딕셔너리 컴프리헨션 · <code>dict(zip())</code></td><td>Python 기본 문법 · 내장 함수</td><td>묶음 → 딕셔너리</td><td>1-2 <code>{k: x * y for k, x, y in zip(a, b, c)}</code></td></tr>
<tr><td><code>strict=True</code> · <code>ValueError</code></td><td>zip 인자 · 예외</td><td>길이 검사</td><td>2-1 <code>zip(..., strict=True)</code></td></tr>
<tr><td><code>enumerate()</code> · <code>zip(*pairs)</code></td><td>Python 내장 함수</td><td>번호 붙이기 · 다시 풀기</td><td>2-2 <code>names, counts = zip(*pairs)</code></td></tr>
<tr><td><code>sorted(set())</code></td><td>Python 내장 함수</td><td>중복 제거 + 정렬</td><td>3 <code>classes = sorted(set(labels))</code></td></tr>
</tbody></table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">✅ 할 것</th><th style="text-align: left;">⚠️ 하지 말 것</th></tr>
</thead>
<tbody>
<tr><td>짝이 꼭 맞아야 하면 <code>strict=True</code></td><td>길이 확인 없이 <code>zip</code></td></tr>
<tr><td>확인할 때 <code>list(zip(...))</code></td><td>zip 객체를 두 번 돌리기</td></tr>
<tr><td>클래스 번호는 <code>sorted(set())</code>로</td><td><code>set()</code> 순서에 번호 맡기기</td></tr>
</tbody></table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">짝</th><th style="text-align: left;">차이</th><th style="text-align: left;">결론</th></tr>
</thead>
<tbody>
<tr><td><code>zip()</code> vs <code>strict=True</code></td><td>잘림 vs 에러</td><td>데이터 짝은 strict</td></tr>
<tr><td><code>zip(a, b)</code> vs <code>zip(*pairs)</code></td><td>묶기 vs 풀기</td><td>방향이 반대</td></tr>
<tr><td><code>dict(zip())</code> vs 컴프리헨션</td><td>그대로 짝 vs 계산해서</td><td>계산이 있으면 컴프리헨션</td></tr>
<tr><td><code>enumerate</code> vs <code>zip</code></td><td>번호 + 값 vs 값 + 값</td><td>둘 다면 <code>enumerate(zip())</code></td></tr>
</tbody></table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">언제</th><th style="text-align: left;">할 일</th><th style="text-align: left;">걸리는 시간</th></tr>
</thead>
<tbody>
<tr><td>내일</td><td>기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td><td>10분</td></tr>
<tr><td>3일 뒤</td><td>변형 문제 3개 다시 풀기 · 원본 정답 세 방법을 직접 쓰기</td><td>15분</td></tr>
<tr><td>7일 뒤</td><td>3장에 라벨 하나를 빼고 strict 에러를 직접 확인</td><td>20분</td></tr>
<tr><td>14일 뒤</td><td>오답 노트에 남은 문제만 · 02_Python 클래스 파일별 1 시작 전 ✅ 줄만 읽기</td><td>10분</td></tr>
</tbody></table>

---
## 부록

### 이 노트북을 만든 방법

- 1장 · 원본 문제(`24-1.Parallel Iteration_Q.ipynb`)의 세 가지 방법
- 2장 · 힌트의 `strict` · `*iterables`를 길이 검사 · 풀기로 확장
- 모든 코드는 실행한 출력과 함께 저장 · 형식 v2.1 (FORMAT_V2.md 1 ~ 7절)

### 읽는 법

*표시 → 의미*

- **`A` · `B`** · **A** 딥러닝 코드에 바로 나오는 필수 · **B** 알아두면 되는 것
- **🎯** · 이번 섹션에서 답해야 할 질문
- **💡** · 쉬운 설명 · 비유
- **⚠️** · 자주 하는 실수
- **✅** · 기억할 결론 = 🎯 질문의 답

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>문제 · 리스트 3개</td><td>zip이 만드는 묶음 보기</td><td>1-1</td></tr>
<tr><td>문제 · 여러 방법 · 출력 결과</td><td>정답 세 가지 방법</td><td>1-2</td></tr>
<tr><td>힌트 · strict=False</td><td>길이가 다를 때 · strict</td><td>2-1</td></tr>
<tr><td>힌트 · *iterables</td><td>번호와 함께 · 다시 풀기</td><td>2-2</td></tr>
<tr><td>힌트 · 묶어 계산하는 과정</td><td>0장 원본 힌트 · 3장 이미지 · 라벨</td><td>0장 · 3</td></tr>
</tbody></table>

### 전체 코드

1 ~ 3장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

**실행 흐름**

1. 1-1 세 리스트를 `zip()`으로 묶기
2. 1-2 이름별 금액을 세 방법으로 만들기
3. 세 결과가 같은지 확인하고 출력

In [9]:
# ---------- 1-1 ----------
# 왜 · 같은 위치끼리 짝이 맞는 세 리스트를 원본 문제 그대로 준비하기 위해
a = ["one", "two", "three", "four"]  # 키가 될 이름
b = [30, 20, 15, 75]                 # 수량
c = [5.2, 7.4, 3.6, 4.2]             # 단가


# 왜 · zip이 같은 위치끼리 묶는다는 것을 list로 펼쳐 눈으로 확인하기 위해
pairs = list(zip(a, b, c))              # 같은 위치끼리 (이름, 수량, 단가)
print(len(pairs), pairs[0], pairs[-1])  # 묶음 수 · 처음 · 마지막
print(type(zip(a, b)))                  # list로 감싸지 않으면 내용이 안 보임


# ---------- 1-2 ----------
# 방법 1: 딕셔너리 컴프리헨션
# 왜 · 세 리스트를 한 줄로 묶어 이름별 금액을 바로 만들기 위해
r1 = {k: x * y for k, x, y in zip(a, b, c)}  # 묶음마다 이름: 수량 × 단가


# 방법 2: 반복문으로 하나씩 넣기
# 왜 · 컴프리헨션이 하는 일을 단계별로 풀어 비교하기 위해
r2 = {}                       # 빈 딕셔너리
for k, x, y in zip(a, b, c):  # 묶음 풀기
    r2[k] = x * y             # 키 = 값으로 추가


# 방법 3: 금액 리스트를 먼저 만들고 dict(zip(키, 값))
# 왜 · 키 리스트와 값 리스트를 따로 만든 뒤 합치는 방식도 비교하기 위해
r3 = dict(zip(a, [x * y for x, y in zip(b, c)]))  # 금액 리스트 → 이름과 짝


# 왜 · 세 방법의 결과가 같다는 것을 코드로 확인하기 위해
assert r1 == r2 == r3  # 세 방법 결과가 같은지
print(r1)              # 원본 출력 결과

4 ('one', 30, 5.2) ('four', 75, 4.2)
<class 'zip'>
{'one': 156.0, 'two': 148.0, 'three': 54.0, 'four': 315.0}


**실행 흐름**

1. 2-1 길이가 다른 `zip()`과 `strict=True`
2. 2-2 `enumerate()` · `zip(*)`으로 풀기
3. 블록마다 결과 출력

In [10]:
# ---------- 2-1 ----------
# 왜 · 길이가 다르면 zip이 조용히 잘라 버리는 위험을 strict=True로 잡기 위해
print(list(zip([1, 2, 3], ["x", "y"])))            # 짧은 쪽에 맞춰 조용히 잘림
try:
    list(zip([1, 2, 3], ["x", "y"], strict=True))  # 길이가 다르면 에러로 알림
except ValueError as e:                            # Python 3.10 이상
    print("ValueError:", e)                        # 에러 문장 출력


# ---------- 2-2 ----------
# 왜 · 묶음 번호가 필요할 때 enumerate와 zip을 함께 쓰는 법을 보이기 위해
for i, (k, x) in enumerate(zip(a, b), start=1):  # 번호 + (이름, 수량)
    print(i, k, x)                               # 몇 번째 묶음인지


# 왜 · zip(*묶음)으로 묶음을 다시 줄별로 푸는 방법을 보이기 위해
pairs = list(zip(a, b))      # [('one', 30), ...]
names, counts = zip(*pairs)  # *로 묶음을 풀어 다시 zip → 줄별로
print(names, counts)         # 튜플 두 개

[(1, 'x'), (2, 'y')]
ValueError: zip() argument 2 is shorter than argument 1
1 one 30
2 two 20
3 three 15
4 four 75
('one', 'two', 'three', 'four') (30, 20, 15, 75)


**실행 흐름**

1. 경로 · 라벨 준비
2. 클래스 이름을 정렬해 번호 매기기
3. 길이를 검사하며 묶기
4. 번호 붙여 출력하고 다시 풀기

In [11]:
# ---------- 3. 종합 실습 ----------
# 1단계: 가짜 이미지 경로와 라벨 (실제로는 폴더에서 읽음)
# 왜 · 이미지 데이터셋을 만드는 실제 상황을 작은 목록으로 재현하기 위해
paths = ["img/cat1.jpg", "img/dog1.jpg", "img/cat2.jpg", "img/bird1.jpg"]  # 이미지 경로
labels = ["cat", "dog", "cat", "bird"]                                     # 같은 위치의 정답


# 2단계: 클래스 이름 → 번호 (정렬해서 항상 같은 번호)
# 왜 · 실행할 때마다 클래스 번호가 바뀌지 않게 하기 위해
classes = sorted(set(labels))                           # 중복 제거 후 정렬
class_to_idx = dict(zip(classes, range(len(classes))))  # 이름과 번호 짝
print(class_to_idx)                                     # {'bird': 0, 'cat': 1, 'dog': 2}


# 3단계: 길이를 검사하며 (경로, 번호) 묶기
# 왜 · 경로와 라벨 수가 다르면 바로 에러로 알기 위해
data = [(p, class_to_idx[y]) for p, y in zip(paths, labels, strict=True)]  # 짝이 안 맞으면 에러


# 4단계: 번호 붙여 출력하고 다시 풀기
# 왜 · 만든 (경로, 번호) 묶음을 확인하고 다시 두 목록으로 되돌리기 위해
for i, (p, y) in enumerate(data):  # 0부터 샘플 번호
    print(i, p, y)                 # 경로와 라벨 번호
xs, ys = zip(*data)                # 경로 줄 · 라벨 줄로 나누기
print("라벨 줄:", ys)              # 모델에 넣을 정답

{'bird': 0, 'cat': 1, 'dog': 2}
0 img/cat1.jpg 1
1 img/dog1.jpg 2
2 img/cat2.jpg 1
3 img/bird1.jpg 0
라벨 줄: (1, 2, 1, 0)
